In [1]:
import pandas as pd 
import numpy as np 
import requests
import boto3
import pg8000
import json
import os 
import requests
from PIL import Image
from io import BytesIO
import math
from scipy.ndimage import label
import io
from skimage.color import rgb2lab
from pg8000.dbapi import DatabaseError, ProgrammingError
from dotenv import load_dotenv
from datetime import datetime, timedelta, time
import gc
import time
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))
from real_time_weather_ingestion import main_scrapper


# --- S3 SETUP ---
BUCKET_NAME = "dsa3101-storm-tracking-tw08"

# create s3 client (this reads credentials from ~/.aws/credentials)
s3 = boto3.client("s3")


# helpers

In [2]:
####################################################################################################
# helpers 
def get_secret(secret_arn):
    resp = secrets_client.get_secret_value(SecretId=secret_arn)
    return json.loads(resp['SecretString'])

def get_db_conn(secret_arn):
    global _db_conn
    if _db_conn:
        try:
            cur = _db_conn.cursor()
            cur.execute("SELECT 1;")
            cur.close()
            return _db_conn
        except Exception:
            _db_conn = None
    secret = get_secret(secret_arn)
    host = secret['host']
    dbname = secret['dbname']
    user = secret['username']
    password = secret['password']
    port = int(secret.get('port', 5432))
    _db_conn = pg8000.connect(
        host=host,
        database=dbname,
        user=user,
        password=password,
        port=port
    )
    return _db_conn

def query_to_df(conn, query, params=None):
    """
    Execute an SQL query and return results as a pandas DataFrame.

    Args:
        conn : active pg8000 connection
        query : SQL query string (use %s placeholders)
        params : tuple/list of parameters (optional)
    """
    with conn.cursor() as cur:
        cur.execute(query, params or ())
        # Extract column names from cursor description
        columns = [desc[0] for desc in cur.description]
        data = cur.fetchall()
    # Create DataFrame
    return pd.DataFrame(data, columns=columns)





In [12]:
# Load environment variables from .env file
load_dotenv()

# Read credentials
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 5432))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")

# Create connection
conn = pg8000.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)

print("✅ Connected successfully to PostgreSQL!")

✅ Connected successfully to PostgreSQL!


In [5]:
def fetch_s3_bytes(s3_key):
    try:
        obj = s3.get_object(Bucket=BUCKET_NAME, Key=s3_key)
        return obj['Body'].read()   # byte data
    except Exception as e:
        print(f"Error fetching {s3_key}: {e}")
        return None

# main backfil

In [8]:
query = """
    SELECT
        distinct(timestamp::date) as date_range

    FROM weather_observation 
    order by 1 asc
    """
    
df = query_to_df(conn,query)

# convert to datetime
df['date_converted'] = pd.to_datetime(df['date_range'])

# lead the converted date by 1 (next row)
df['next_date'] = df['date_converted'].shift(-1)

# compute the day difference
df['diff_days'] = (df['next_date'] - df['date_converted']).dt.days


# filter rows with gaps > 1 day
gap_df = df[df['diff_days'] > 1]

gap_df


,date_range,date_converted,next_date,diff_days
1,2025-07-04,2025-07-04,2025-08-01,28.0
6,2025-08-05,2025-08-05,2025-08-16,11.0


In [5]:
query = """
    SELECT
        *
    FROM weather_observation

    """
    
df = query_to_df(conn,query)
df


,obs_id,station_id,timestamp,wind_speed,wind_direction,rainfall_mm,temperature_c,humidity_pct
0,103858,S109,2025-10-08 00:00:00,1.0,191.0,0.0,28.0,81.7
1,103859,S106,2025-10-08 00:00:00,0.6,33.0,0.0,26.6,92.9
2,103860,S117,2025-10-08 00:00:00,3.0,133.0,0.0,27.7,87.0
3,103861,S107,2025-10-08 00:00:00,7.3,213.0,0.0,28.7,77.6
4,103862,S115,2025-10-08 00:00:00,1.8,135.0,0.0,28.4,84.2
...,...,...,...,...,...,...,...,...
174468,187314,S50,2025-10-22 23:55:00,3.2,315.0,0.0,26.9,76.5
174469,187315,S44,2025-10-22 23:55:00,3.3,4.0,0.0,26.3,82.2
174470,187316,S43,2025-10-22 23:55:00,2.0,255.0,0.0,27.5,75.4
174471,187317,S111,2025-10-22 23:55:00,4.7,245.0,0.0,26.5,77.4


In [ ]:
# conn.rollback()

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

# day range 
start_date_str = "2025-08-21"
# end_date_str   = "2025-10-25"
end_date_str   = "2025-08-31"

start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
end_date   = datetime.strptime(end_date_str, "%Y-%m-%d").date()

MAX_WORKERS = 3

current_date = start_date
while current_date <= end_date:
    run_date_str = current_date.strftime('%Y-%m-%d')
    print(f'processing weather scrapes for full day: {run_date_str}')

    # build 5-min timeline for the entire day
    timeline = []
    day_start = datetime.combine(current_date, datetime.min.time())
    for h in range(24):
        for m in range(0, 60, 5):
            ts = day_start + timedelta(hours=h, minutes=m)
            timeline.append(ts.isoformat())

    # parallel block: run main_scrapper() for each timestamp
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(main_scrapper, ts): ts for ts in timeline}
        for fut in as_completed(futures):
            ts = futures[fut]
            try:
                df = fut.result()
                if df is not None and not df.empty:
                    results.append(df)
                    print(f" {ts} done, {df.shape[0]} rows")
                else:
                    print(f" {ts} empty or None")
            except Exception as e:
                print(f" {ts} failed: {e}")

    # combine all df for the day

    combined_day_df = pd.concat(results, ignore_index=True)
    combined_day_df = combined_day_df.rename(columns={
    'wind_direction_degrees': 'wind_direction',
    'wind_speed_knots': 'wind_speed',
    'time_stamp': 'timestamp'
    })
    print(f" combined day {run_date_str} shape: {combined_day_df.shape}")


    # build sql insert
    insert_sql = """
    INSERT INTO weather_observation (
        timestamp, station_id, wind_direction, wind_speed,
        rainfall_mm, temperature_c, humidity_pct
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (timestamp, station_id)
    DO UPDATE SET
        wind_direction         = EXCLUDED.wind_direction,
        wind_speed             = EXCLUDED.wind_speed,
        rainfall_mm            = EXCLUDED.rainfall_mm,
        temperature_c          = EXCLUDED.temperature_c,
        humidity_pct           = EXCLUDED.humidity_pct;
    """

    # prepare rows for executemany
    rows = list(
        combined_day_df[
            ['timestamp','station_id','wind_direction','wind_speed',
             'rainfall_mm','temperature_c','humidity_pct']
        ].itertuples(index=False, name=None)
    )

    # batch upload to postgresql
    try:
        print(f'running insert for {run_date_str}')
        start = time.time()
        with conn.cursor() as cur:
            cur.executemany(insert_sql, rows)
        conn.commit()
        end = time.time()
        elapsed_min = (end - start) / 60
        print(f" uploaded {len(rows)} rows for {run_date_str}, upload time : {elapsed_min}")
    except Exception as e:
        conn.rollback()
        print(f" db upload failed for {run_date_str}: {e}")
        raise

    # move to next day
    current_date += timedelta(days=1)


processing weather scrapes for full day: 2025-08-21
 2025-08-21T00:00:00 done, 11 rows
 2025-08-21T00:05:00 done, 11 rows
 2025-08-21T00:10:00 done, 11 rows
 2025-08-21T00:15:00 done, 11 rows
 2025-08-21T00:20:00 done, 11 rows
 2025-08-21T00:25:00 done, 11 rows
 2025-08-21T00:30:00 done, 11 rows
 2025-08-21T00:35:00 done, 11 rows
 2025-08-21T00:40:00 done, 11 rows
 2025-08-21T00:45:00 done, 11 rows
 2025-08-21T00:50:00 done, 10 rows
 2025-08-21T00:55:00 done, 11 rows
 2025-08-21T01:00:00 done, 11 rows
 2025-08-21T01:05:00 done, 11 rows
 2025-08-21T01:10:00 done, 11 rows
 2025-08-21T01:15:00 done, 11 rows
 2025-08-21T01:20:00 done, 11 rows
 2025-08-21T01:25:00 done, 11 rows
 2025-08-21T01:30:00 done, 11 rows
 2025-08-21T01:35:00 done, 11 rows
 2025-08-21T01:40:00 done, 11 rows
 2025-08-21T01:45:00 done, 11 rows
 2025-08-21T01:50:00 done, 11 rows
 2025-08-21T01:55:00 done, 11 rows
 2025-08-21T02:00:00 done, 11 rows
 2025-08-21T02:05:00 done, 11 rows
 2025-08-21T02:10:00 done, 11 rows
 20

# archive

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

# day range 
start_date_str = "2025-08-02"
# end_date_str   = "2025-10-25"
end_date_str   = "2025-08-03"

start_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
end_date   = datetime.strptime(end_date_str, "%Y-%m-%d").date()


current_date = start_date
while current_date <= end_date:
    day_df = pd.DataFrame()
    run_date_str = current_date.strftime('%Y-%m-%d')
    print(f'processing weather scrapes for full day: {run_date_str}')

    # build 5-min timeline for the entire day
    timeline = []
    day_start = datetime.combine(current_date, datetime.min.time())
    for h in range(2):
        for m in range(0, 60, 5):
            ts = day_start + timedelta(hours=h, minutes=m)
            timeline.append(ts.isoformat(sep=' '))


    for ts in timeline:
        curr_df = main_scrapper(ts)
        combined_day_df = pd.concat([day_df,curr_df],ignore_index=True)

    
    print(f" combined day {run_date_str} shape: {combined_day_df.shape}")

    # build sql insert
    insert_sql = """
    INSERT INTO weather_observation (
        timestamp, station_id, wind_direction, wind_speed,
        rainfall_mm, temperature_c, humidity_pct
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (timestamp, station_id)
    DO UPDATE SET
        wind_direction = EXCLUDED.wind_direction,
        wind_speed       = EXCLUDED.wind_speed,
        rainfall_mm            = EXCLUDED.rainfall_mm,
        temperature_c          = EXCLUDED.temperature_c,
        humidity_pct           = EXCLUDED.humidity_pct;
    """

    # prepare rows for executemany
    rows = list(
        combined_day_df[
            ['time_stamp','station_id','wind_direction','wind_speed',
             'rainfall_mm','temperature_c','humidity_pct']
        ].itertuples(index=False, name=None)
    )

    # batch upload to postgresql
    try:
        with conn.cursor() as cur:
            cur.executemany(insert_sql, rows)
        conn.commit()
        print(f" uploaded {len(rows)} rows for {run_date_str}")
    except Exception as e:
        conn.rollback()
        print(f" db upload failed for {run_date_str}: {e}")
        raise

    # move to next day
    current_date += timedelta(days=1)
